Optimization domain of the Rastrigin function: $[-5.12, 5.12]{^2}$

In [20]:
import numpy as np

class CMAES:
  def __init__(self, dim, func) -> None:
    self.func = func
    self.dim = dim
    self.l = int(4 + np.floor(3 * np.log(dim)))
    self.mu = self.l // 2
    self.w_temp = np.array([np.log((self.l + 1) / 2) - np.log(i) for i in range(1, self.l + 1)])
    self.mu_eff = np.pow(np.sum(self.w_temp[:self.mu]), 2) / np.sum(np.pow(self.w_temp[:self.mu], 2))
    self.alpha_cov = 2
    self.c_m = 1
    self.c1 = self.alpha_cov / (np.pow(dim + 1.3, 2) + self.mu_eff)
    self.cc = (4 + self.mu_eff / dim) / (dim + 4 + 2 * (self.mu_eff / dim))
    self.c_mu = np.min([1 - self.c1, self.alpha_cov * (1/4 + self.mu_eff + (1 / self.mu_eff) - 2) / (np.pow(dim + 2, 2) + self.alpha_cov * (self.mu_eff / 2))])
    self.c_sigma = (self.mu_eff + 2) / (dim + self.mu_eff + 5)
    self.d_sigma = 1 + 2 * np.max([0, np.sqrt((self.mu_eff - 1) / (dim + 1)) - 1]) + self.c_sigma
    self.alpha_mu_neg = 1 + (self.c1 / self.c_mu)
    self.mu_neg_eff = (np.pow(np.sum(self.w_temp[self.mu : self.l]), 2)) / (np.sum(np.pow(self.w_temp[self.mu : self.l], 2)))
    self.alpha_mu_neg_eff = 1 + (2 * self.mu_neg_eff) / (self.mu_eff + 2)
    self.alpha_neg_pos_def = (1 - self.c1 - self.c_mu) / (dim * self.c_mu)
    self.w = np.zeros_like(self.w_temp)
    for i in range(len(self.w)):
      if self.w_temp[i] >= 0:
        self.w[i] = (1 / (np.sum(np.abs(np.where(self.w_temp >= 0, self.w_temp, 0))))) * self.w_temp[i]
      else:
        self.w[i] = (np.min([self.alpha_mu_neg, self.alpha_mu_neg_eff, self.alpha_neg_pos_def]) / np.sum(np.abs(np.where(self.w_temp < 0, self.w_temp, 0)))) * self.w_temp[i]

  def evaluate(self, specimen):
    evaluated = self.func(specimen)
    return np.argsort(evaluated)

  def optimize(self, max_iter, domain_min, domain_max):
    # Initialization
    p_sigma = np.zeros(self.dim)
    p_c = np.zeros(self.dim)
    C = np.identity(self.dim)
    g = 0
    sigma = 0.3 * (domain_max - domain_min)
    m = np.random.uniform(domain_min, domain_max, size=self.dim)
    t = 1
    iter = 0
    x_best = None
    best_value = np.inf
    # Until termination criterion met
    while iter < max_iter:
      g += 1
      # Sample new population of search points, for k = 1, ... , lambda
      Z = np.random.normal(0, 1, size=(self.dim, self.l))
      if not np.all(np.isfinite(C)):
        print("Niepoprawne C w pokoleniu:", g)
        print(C)
        break
      eigenvalues, B = np.linalg.eigh(C)

      # We need to stop the algorithm when the cloud becomes too small
      # Because then, we encounter FP precision errors that prevent
      # The matrix from being positive definite
      if sigma * np.sqrt(eigenvalues[0]) < 1e-10:
        break

      D = np.identity(self.dim) * np.sqrt(eigenvalues)
      Y = B @ D @ Z
      X = m[:, None] + sigma * Y

      # Selection and recombination
      evaluated_indices = self.evaluate(X)
      Y_evaluated = Y[:, evaluated_indices]
      Y_best = Y_evaluated[:, :self.mu]
      Y_w = np.sum(self.w[:self.mu]*Y_best, axis=1)
      m = m + self.c_m * sigma * Y_w

      # Step-size control
      C_pow_neg_half = B @ np.diag(1 / np.sqrt(eigenvalues)) @ np.transpose(B)
      p_sigma = (1 - self.c_sigma) * p_sigma + np.sqrt(self.c_sigma * (2 - self.c_sigma) * self.mu_eff) * C_pow_neg_half @ Y_w
      E_z_norm = np.sqrt(self.dim) * (1 - (1 / (4 * self.dim)) + (1 / (21 * np.pow(self. dim, 2))))
      sigma = sigma * np.exp((self.c_sigma / self.d_sigma) * (np.linalg.norm(p_sigma) / E_z_norm -1 ))

      # Covariance matrix adaptation
      h_sigma = 1 if np.linalg.norm(p_sigma) / np.sqrt(1 - np.pow(1 - self.c_sigma, 2 * g)) < (1.4 + 2 / (self.dim + 1)) * E_z_norm else 0
      p_c = (1 - self.cc) * p_c + h_sigma * np.sqrt(self.cc * (2 - self.cc) * self.mu_eff) * Y_w
      C_update_weights = np.zeros_like(self.w)
      for i in range(len(self.w)):
        if self.w[i] < 0:
          C_update_weights[i] = self.w[i] * (self.dim / np.pow(np.linalg.norm(C_pow_neg_half @ Y_evaluated[:, i]), 2))
        else:
          C_update_weights[i] = self.w[i]
      delta_h_sigma = (1 - h_sigma) * self.cc * (2 - self.cc)
      C = (1 + self.c1 * delta_h_sigma - self.c1 - self.c_mu * np.sum(self.w)) * C + self.c1 * p_c[:, None] @ p_c[None, :] + self.c_mu * (C_update_weights * Y_evaluated) @ np.transpose(Y_evaluated)
      candidate = X[:, evaluated_indices[0]]
      candidate_value = self.func(candidate)
      if candidate_value < best_value:
        best_value = candidate_value
        x_best = candidate.copy()
      iter += 1

    return x_best


In [23]:
def rastrigin_function(X):
  return 10 * X.shape[0] + np.sum(np.pow(X, 2) - 10 * np.cos(2 * np.pi * X), axis=0)

In [24]:
import plotly.graph_objects as go

X = np.mgrid[-5:5:0.25, -5:5:0.25]
Z = rastrigin_function(X)

fig = go.Figure(
    data=[go.Surface(x=X[0], y=X[1], z=Z, colorscale="Viridis")]
)

fig.update_layout(
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="f(X,Y)"
    ),
    height=650
)

cmaes = CMAES(3, rastrigin_function)
X_best = cmaes.optimize(10000, -5, 5)

fig.add_trace(go.Scatter3d(
    x=[X_best[0]],
    y=[X_best[1]],
    z=[rastrigin_function(X_best)],
    mode="markers",
    marker=dict(size=6, color="red"),
    name="Najlepsze rozwiązanie"
))

fig.show(renderer="colab")